# Control Variable Analysis

Objective:
- Load the scored CSV.
- Merge control variables (market cap, firm age, etc.) from WRDS if needed.
- Run regressions of clarity against controls to check if clarity is just proxying for firm characteristics.

### Load scored data and merge controls

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/interim/buyback_scored.csv')
print(f"Loaded {len(df)} scored transcripts")

DATE_COL = 'announceddate'  # Adapt based on dataset columns
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
df['year'] = df[DATE_COL].dt.year

# Check what control variables are already in the data
control_candidates = ['market_cap', 'mktcap', 'log_mktcap', 'firm_age',
                      'age', 'analyst_coverage', 'num_analysts',
                      'leverage', 'total_debt', 'revenue', 'assets']
available = [c for c in df.columns if any(cc in c.lower() for cc in control_candidates)]
print(f"Potential control columns found: {available}")


Loaded 11313 scored transcripts
Potential control columns found: []


### Fetch Missing Controls from WRDS (If Necessary)

In [12]:
CONTROL_CSV_PATH = '../data/interim/buyback_controls.csv'
REQUIRED_CONTROLS = ['mktcap', 'leverage', 'firm_age']

if set(REQUIRED_CONTROLS).issubset(set(available)):
    print("All required controls are already present in the dataset.")
elif os.path.exists(CONTROL_CSV_PATH):
    print("Loading previously downloaded control variables...")
    controls_df = pd.read_csv(CONTROL_CSV_PATH).drop_duplicates(subset=['companyid', 'year'])
    # Merge on companyid and year, assuming controls_df has them
    if 'companyid' in controls_df.columns and 'year' in controls_df.columns:
        df = df.merge(controls_df, on=['companyid', 'year'], how='left')
        available.extend([c for c in controls_df.columns if c not in available])
        print(f"Merged controls. Current control columns: {available}")
    else:
        print("Control CSV is missing required merge keys ('companyid', 'year').")
else:
    print("Fetching missing controls from WRDS...")
    try:
        import wrds
        db = wrds.Connection()  # Assuming WRDS_USERNAME is in env or pgpass is set up

        # 1. Map Capital IQ companyid to Compustat GVKEY
        # ciq.wrds_gvkey maps CIQ companyid to GVKEY
        ciq_ids = tuple(df['companyid'].dropna().astype(int).unique())
        print(f"Fetching GVKEY map for {len(ciq_ids)} CIQ companies...")
        
        map_query = f"""
            SELECT companyid, gvkey
            FROM ciq.wrds_gvkey
            WHERE companyid IN {ciq_ids}
        """
        id_map = db.raw_sql(map_query)
        id_map['companyid'] = id_map['companyid'].astype(int)
        df = df.merge(id_map, on='companyid', how='left')

        gvkeys = tuple(df['gvkey'].dropna().unique())

        # 2. Pull Fundamentals from Compustat (Annual)
        # We'll calculate leverage (dltt + dlc) / at
        print("Fetching Compustat fundamentals (Assets, Debt)...")
        funda_query = f"""
            SELECT gvkey, fyear, at, dltt, dlc, prcc_f, csho
            FROM comp.funda
            WHERE gvkey IN {gvkeys}
              AND fyear >= 2008
              AND indfmt='INDL'
              AND datafmt='STD'
              AND popsrc='D'
              AND consol='C'
        """
        funda = db.raw_sql(funda_query)
        funda['fyear'] = funda['fyear'].astype(int)
        
        # Compute metrics
        # leverage = total debt / total assets
        funda['total_debt'] = funda['dltt'].fillna(0) + funda['dlc'].fillna(0)
        funda['leverage'] = funda['total_debt'] / funda['at'].clip(lower=1)
        # Market cap proxy from Compustat if CRSP isn't merged
        funda['mktcap'] = funda['prcc_f'] * funda['csho']
        
        # Get firm age (proxy: years since first appearance in Compustat)
        print("Fetching firm age...")
        age_query = f"""
            SELECT gvkey, MIN(fyear) as first_year
            FROM comp.funda
            WHERE gvkey IN {gvkeys}
            GROUP BY gvkey
        """
        age_df = db.raw_sql(age_query)
        funda = funda.merge(age_df, on='gvkey', how='left')
        funda['firm_age'] = funda['fyear'] - funda['first_year']

        # Prepare final controls dataframe
        controls_df = funda[['gvkey', 'fyear', 'mktcap', 'leverage', 'firm_age']].rename(columns={'fyear': 'year'})
        controls_df = controls_df.dropna(subset=['gvkey'])
        
        # We also need companyid back on the controls dataframe to save it properly
        controls_df = controls_df.merge(id_map, on='gvkey', how='inner')
        controls_df = controls_df.drop_duplicates(subset=['companyid', 'year'])
        
        # Save controls to disk to avoid re-pulling
        controls_df.to_csv(CONTROL_CSV_PATH, index=False)
        print(f"Saved control variables to {CONTROL_CSV_PATH}")

        # Merge onto main dataframe
        df = df.merge(controls_df, on=['companyid', 'year'], how='left')
        
        available.extend(['mktcap', 'leverage', 'firm_age'])
        print("Successfully merged WRDS control variables.")
        
    except ImportError:
        print("Error: The `wrds` Python package is not installed. Run `pip install wrds` to fetch controls.")
    except Exception as e:
        print(f"Failed to fetch from WRDS: {e}")

Loading previously downloaded control variables...
Merged controls. Current control columns: ['gvkey', 'year', 'mktcap', 'leverage', 'firm_age', 'companyid']


### Regress clarity against controls

In [13]:
# Check if clarity is just proxying for firm size, age, etc.
# OLS: clarity_composite ~ log(market_cap) + firm_age + analyst_coverage + sector_FE

if 'clarity_composite' in df.columns:
    y = df['clarity_composite'].dropna()

    # Build X matrix from available controls
    X_cols = []
    
    if 'mktcap' in df.columns:
        df['log_mktcap'] = np.log(df['mktcap'].clip(lower=1))
        X_cols.append('log_mktcap')
    if 'firm_age' in df.columns:
        X_cols.append('firm_age')
    if 'leverage' in df.columns:
        X_cols.append('leverage')

    if X_cols:
        # Align index for valid regression (drop NA rows for X and y simultaneously)
        valid_idx = df[X_cols + ['clarity_composite']].dropna().index
        y_valid = df.loc[valid_idx, 'clarity_composite'].astype(float)
        X_valid = df.loc[valid_idx, X_cols].astype(float)
        
        X_valid = sm.add_constant(X_valid)
        model = sm.OLS(y_valid, X_valid).fit()
        print(model.summary())
        print(f"\nR-squared: {model.rsquared:.4f}")
        print("If R-squared is high (>0.15), clarity may be proxying for firm characteristics.")
        print("If R-squared is low (<0.05), clarity captures independent information.")
    else:
        print("No valid control columns found to run regression.")
else:
    print("Clarity composite column not found. Did Notebook 07 run successfully?")

                            OLS Regression Results                            
Dep. Variable:      clarity_composite   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     43.19
Date:                Sat, 11 Apr 2026   Prob (F-statistic):           9.68e-28
Time:                        13:44:19   Log-Likelihood:                -8632.3
No. Observations:               11190   AIC:                         1.727e+04
Df Residuals:                   11186   BIC:                         1.730e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.2209      0.023     -9.496      0.0

### Explore additional controls

In [14]:
# Additional regressions to explore:
# 1. Does sentiment correlate with firm size?
# 2. Does hedge density correlate with industry?
# 3. Does FOG correlate with firm age?
# 4. Does the sentiment gap correlate with revenue surprise?

# These regressions help determine which controls are needed
# in the final event study specification

# Print correlation matrix of all metrics vs available controls
if available:
    metrics = ['buyback_sentiment_mean', 'clarity_composite', 'specificity',
               'hedge_density', 'modified_fog']
    if 'qa_relevance' in df.columns and df['qa_relevance'].notna().sum() > 100:
        metrics.append('qa_relevance')
        
    valid_metrics = [m for m in metrics if m in df.columns]
    valid_controls = [c for c in available if c in df.columns]
    
    if valid_metrics and valid_controls:
        corr = df[valid_metrics + valid_controls].corr()
        print("Correlation: metrics vs controls")
        print(corr.loc[valid_metrics, valid_controls].round(3))

Correlation: metrics vs controls
                        gvkey   year  mktcap  leverage  firm_age  companyid
buyback_sentiment_mean -0.028  0.129   0.026     0.028     0.042      0.042
clarity_composite      -0.065 -0.042   0.008     0.039     0.077     -0.032
specificity            -0.032 -0.059  -0.033     0.092     0.045     -0.012
hedge_density           0.045 -0.006  -0.034    -0.004    -0.056      0.002
modified_fog            0.026  0.013  -0.011     0.034    -0.022      0.036
